# NCBI Virus Dataset Creation

Author: Alexander Maksiaev

Purpose: Create dataset using NCBI and relabeling sequences. 

Notes:
* This file must be in the same folder as "utils.py"

## Housekeeping

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
import importlib
import utils  
importlib.reload(utils)
from utils import * # If changing utils, must restart this file for changes to take effect

In [2]:
# Directory paths and input

# Input
browser = input("Browser (Firefox, Chrome, or Edge): ")
sleep_time = input("Seconds to wait in between clicks (recommended 5): ")
# locations = input("Locations (separate with commas and no spaces in between locations): ")
# start_date = input("Start date (format: MM-DD-YYYY): ")
# end_date = input("End date: (format: MM-DD-YYYY): ")
# serotype = input("Serotype (e.g. H5N1): ")
# serotypes = list(serotype)
# genotypes = input("Genotypes (separate with commas and no spaces in between genotypes): ")
# genotypes = genotypes.split(",")

# Dates and locations
locations = "Antarctica,North America,South America"
start_date = "11-01-2021"
end_date = "01-16-2026"
prev_end_date = "01-09-2026"
date_range = start_date + "--" + end_date
prev_date_range = start_date + "--" + prev_end_date

# Maintenance serotypes and genotypes
serotypes = ["H5N1"]
genotypes = ["B3.13", "D1.1", "D1.3"]

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Downloads/"
# references = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/"

home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"

downloads_saved = home + "NCBI_Virus/downloads/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" 
prev_downloads_saved = home + "NCBI_Virus/downloads/" + prev_date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" 

andersen = home + "Andersen/avian-influenza/metadata/"
temp_files = home + "NCBI_Virus/temp/"
complete_files = home + "NCBI_Virus/complete/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

# All serotypes and genotypes
# serotype = ""
# genotypes_df = pd.read_excel("genotype_key.xlsx")
# genotypes = list(genotypes_df["Genotype"])


## Downloading Data

In [3]:
os.chdir(downloads)

if not os.path.exists(downloads_saved): # checking if the directory exists or not
    os.makedirs(downloads_saved) # if the directory is not present then create it

# Move downloaded files to saved downloads
for dirpath, dirs, files in os.walk(downloads_saved):
    if len(files) != 0:
        break 
    else: # If we don't have any downloaded files
        # Get files
        open_ncbi_virus(browser, sleep_time, locations, start_date, end_date)

        # Re-try 
        for dirpath, dirs, files in os.walk(downloads):
            if len(files) > 0: # If we have any files that need to be moved
                for file in files:
                    file_name = os.path.join(dirpath, file)
                    destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                    try:
                        shutil.move(file_name, destination_path)
                    except:
                        print("Error moving file", file_name)
                        continue 
            break 
    break 

## De-Duplication

In [4]:
# Get metadata
os.chdir(downloads_saved)
metadata = pd.read_csv("sequences.csv")
print(len(metadata))

# Make sure we only have completed sequences -- 8 segments each 

metadata_counts = metadata.groupby(metadata.Isolate, as_index=False).size()
# print(metadata_counts)
metadata_counted = metadata.merge(metadata_counts, on="Isolate")

# Only keep those with size >= 8

metadata_complete_segs = metadata_counted[metadata_counted["size"] >= 8] # May have duplicates
metadata_complete_segs = metadata_complete_segs.drop_duplicates(subset="GenBank_Title", keep="last") # Get rid of duplicate segments

# Now only accept == 8 segments

metadata_segments = metadata_complete_segs[metadata_complete_segs["size"] == 8]
metadata_segments

140400


,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Host,Tissue_Specimen_Source,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size
0,PX880763.1,GenBank,GCA_054514015.1,SRR36538378,SAMN54232216,PRJNA1207547,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Falco peregrinus,NaN,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-11-20,2026-01-16,ssRNA(-),8
1,PX880764.1,GenBank,GCA_054514015.1,SRR36538378,SAMN54232216,PRJNA1207547,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Falco peregrinus,NaN,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-11-20,2026-01-16,ssRNA(-),8
2,PX880765.1,GenBank,GCA_054514015.1,SRR36538378,SAMN54232216,PRJNA1207547,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Falco peregrinus,NaN,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-11-20,2026-01-16,ssRNA(-),8
3,PX880766.1,GenBank,GCA_054514015.1,SRR36538378,SAMN54232216,PRJNA1207547,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Falco peregrinus,NaN,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-11-20,2026-01-16,ssRNA(-),8
4,PX880767.1,GenBank,GCA_054514015.1,SRR36538378,SAMN54232216,PRJNA1207547,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Falco peregrinus,NaN,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-11-20,2026-01-16,ssRNA(-),8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132376,OK205883.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8
132377,OK205884.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8
132378,OK205885.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8
132379,OK205886.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8


## Add sequences to dataframe

In [5]:
# NCBI Virus Naming Convention:
# "Accession|GenBank_Title|Assembly|SRA Accession|BioSample|BioProject|Genotype|Isolate|Geo Location|Host|Collection Date"

# Get sequences and headers together
# headers = []
# isolates = []
# sras = []
# headers_seqs = {}

os.chdir(downloads_saved)

sequences_fasta = df_from_fasta("sequences.fasta") # Turn fasta into a dataframe

sequences_fasta["Accession"] = sequences_fasta["full_header"].apply(lambda x: x.split(" |")[0].replace(">",""))

print(sequences_fasta["full_header"])

# Extract segment number so that we can add the correct sequences to the correct sample
# sequences_fasta["Segment"] = sequences_fasta["full_header"].apply(lambda x: int(re.search(r'segment (.?) ', x.split("|")[-2]).group(1)))

# Double-check the de-duplication
print(len(sequences_fasta)) 
# print(sequences_fasta.head())
print(len(metadata_segments))

# Add sequences to the dataframe
metadata_segments = pd.merge(metadata_segments, sequences_fasta, on="Accession") # , "Segment"])

0         >PX879581.1 |Influenza A virus (A/Glaucous gul...
1         >PX879582.1 |Influenza A virus (A/Glaucous gul...
2         >PX879583.1 |Influenza A virus (A/Glaucous gul...
3         >PX879584.1 |Influenza A virus (A/Glaucous gul...
4         >PX879585.1 |Influenza A virus (A/Glaucous gul...
                                ...                        
140395    >OK205883.1 |Influenza A virus (A/chicken/Vera...
140396    >OK205884.1 |Influenza A virus (A/chicken/Vera...
140397    >OK205885.1 |Influenza A virus (A/chicken/Vera...
140398    >OK205886.1 |Influenza A virus (A/chicken/Vera...
140399    >OK205887.1 |Influenza A virus (A/chicken/Vera...
Name: full_header, Length: 140400, dtype: object
140400
121096


In [6]:
metadata_segments

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size,full_header,sequence
0,PX880763.1,GenBank,GCA_054514015.1,SRR36538378,SAMN54232216,PRJNA1207547,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-11-20,2026-01-16,ssRNA(-),8,>PX880763.1 |Influenza A virus (A/Peregrine Fa...,ATGGATAGAATAAAAGAACTGAGAGATCTAATGTCACAGTCTCGCA...
1,PX880764.1,GenBank,GCA_054514015.1,SRR36538378,SAMN54232216,PRJNA1207547,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-11-20,2026-01-16,ssRNA(-),8,>PX880764.1 |Influenza A virus (A/Peregrine Fa...,ATGGATGTCAATCCGACTTTACTTTTCTTAAAAGTGCCAGCGCAAG...
2,PX880765.1,GenBank,GCA_054514015.1,SRR36538378,SAMN54232216,PRJNA1207547,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-11-20,2026-01-16,ssRNA(-),8,>PX880765.1 |Influenza A virus (A/Peregrine Fa...,ATGGAAGATTTTGTGCGACAATGCTTCAATCCAATGATCGTCGAGC...
3,PX880766.1,GenBank,GCA_054514015.1,SRR36538378,SAMN54232216,PRJNA1207547,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-11-20,2026-01-16,ssRNA(-),8,>PX880766.1 |Influenza A virus (A/Peregrine Fa...,ATGGAAAACATAGTACTTCTTCTTGCAATAATTAGCCTTGTTAAAA...
4,PX880767.1,GenBank,GCA_054514015.1,SRR36538378,SAMN54232216,PRJNA1207547,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-11-20,2026-01-16,ssRNA(-),8,>PX880767.1 |Influenza A virus (A/Peregrine Fa...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121091,OK205883.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>OK205883.1 |Influenza A virus (A/chicken/Vera...,AGCAAAAGCAGGGGTATCAGATATCAAAATGGAAAGAATAGTGATT...
121092,OK205884.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>OK205884.1 |Influenza A virus (A/chicken/Vera...,AGCAAAAGCAGGTTAGATAATCACTCACCGAGTGACATTCACATCA...
121093,OK205885.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>OK205885.1 |Influenza A virus (A/chicken/Vera...,AGCAAAAGCAGGAGTGAAGATGAATCCAAATCAGAAGATAATAACA...
121094,OK205886.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>OK205886.1 |Influenza A virus (A/chicken/Vera...,AGCAAAAGCAGGTAGATATTGAAAGATGAGTCTTCTAACCGAGGTC...


## Find genotypes

(Using old genoflu results or Andersen Lab genoflu output) <br>
Old genoflu results should accumulate into one file to avoid having to genotype anything again.

### Find old genotypes

In [7]:
# Switch directory to previous week
os.chdir(prev_downloads_saved)

# Get both files from previous week
genoflu_output = pd.read_csv("output.tsv", delimiter="\t")
genoflu_results = pd.read_csv("results.tsv", delimiter="\t")

# Get old results from output.tsv
genoflu_old = pd.concat([genoflu_output, genoflu_results])
# genoflu_old = genoflu_old.rename(columns={"Strain":"Partial_Header"}) # So we can merge
print(genoflu_old)

# Switch back directory
os.chdir(downloads_saved)

# Save this output for the future
genoflu_old.to_csv("output.tsv", sep="\t", index=False)

                                                 Strain  \
0     Influenza_A_virus__Mexico__Estado_de_Mexico_CP...   
1     Influenza_A_virus__Mexico__Estado_de_Mexico_CP...   
2     Influenza_A_virus__Mexico__Ciudad_de_Mexico_CP...   
3     Influenza_A_virus__Mexico__Estado_de_Mexico_CP...   
4     Influenza_A_virus__Mexico__Michoacan_CPA_02011...   
...                                                 ...   
1746  Influenza_A_virus__USA__PA_25_006666_003_origi...   
1747  Influenza_A_virus__USA__IL_25_025606_001_origi...   
1748  Influenza_A_virus__USA__MI_25_013658_010_origi...   
1749  Influenza_A_virus__USA__NY_25_023581_001_origi...   
1750  Influenza_A_virus__USA__MN_25_011610_001_origi...   

                                               Genotype  \
0     Not assigned: Only 3 segments >98.0% match fou...   
1     Not assigned: Only 3 segments >98.0% match fou...   
2     Not assigned: Only 3 segments >98.0% match fou...   
3     Not assigned: Only 3 segments >98.0% match fou...

In [17]:

# metadata_segments["Partial_Header"] = metadata_segments["full_header"].apply(lambda x: x.split("|")[1].split("(")[1]) # Get only genbank name
metadata_segments["Partial_Header_temp"] = metadata_segments["Isolate"] # Get only isolate

# metadata_segments["Partial_Header_temp"] = metadata_segments["Partial_Header"].apply(lambda x: x.replace(">", "").split("segment")[0][:-3]) # Get rid of header indicator, stop at segment bit
print(metadata_segments["Partial_Header_temp"].values[:10])

# metadata_segments["Partial_Header_temp"] = metadata_segments["Partial_Header"].apply(lambda x: x.split("|")[1]) 
for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]: # Forbidden punctuation
    metadata_segments["Partial_Header_temp"] = metadata_segments["Partial_Header_temp"].apply(lambda x: x.replace(c, "_") if x == x else x)




# Merge to get already-genotyped segments
# genoflu_old["Partial_Header_temp"] = genoflu_old["Partial_Header"].apply(lambda x: re.split(r'_H.N._20.{2}_.{2}_.{2}', x)[0]) # Get only genbank name
genoflu_old["Partial_Header_temp"] = genoflu_old["Strain"].apply(lambda x: re.split(r'Influenza_A_virus__\D*__\D*_', x)[-1]) # Get only isolate 
genoflu_old["Partial_Header_temp"] = genoflu_old["Partial_Header_temp"].apply(lambda x: re.split(r'_H.N._20.{2}_.{2}_.{2}', x)[0]) # Get only isolate


print(metadata_segments["Partial_Header_temp"].values[:10])
print(genoflu_old["Partial_Header_temp"].values[100:120])

print(metadata_segments.head())
metadata_segments_old = metadata_segments.merge(genoflu_old, how="inner", on="Partial_Header_temp") 

# Isolate not-already-genotyped segments
metadata_segments_new = metadata_segments.merge(genoflu_old, indicator=True, how='left', on="Partial_Header_temp").loc[lambda x : x['_merge']=='left_only'] 

print(len(metadata_segments_old))
print(len(metadata_segments_new))

['25G04838-001-original' '25G04838-001-original' '25G04838-001-original'
 '25G04838-001-original' '25G04838-001-original' '25G04838-001-original'
 '25G04838-001-original' '25G04838-001-original' '25G05080-001-original'
 '25G05080-001-original']
['25G04838_001_original' '25G04838_001_original' '25G04838_001_original'
 '25G04838_001_original' '25G04838_001_original' '25G04838_001_original'
 '25G04838_001_original' '25G04838_001_original' '25G05080_001_original'
 '25G05080_001_original']
['24_013789_010_original' '1266_8_2022' '22_017571_001_original'
 '22_020657_001_original' '0660_3_2022' '22_012334_001' '0833_79_2022'
 '0097_26_2022' '0014_1_2024' '22_001330_015' '1309_10_2022'
 '22_009329_001' '22_034080_032_original' '0802_90_2022' '0934_84_2022'
 '0481_R' '1466_11_2022' '0100_2_2024' '1628_29_2022'
 '22_028596_002_original']
    Accession GenBank_RefSeq         Assembly SRA_Accession     BioSample  \
0  PX880763.1        GenBank  GCA_054514015.1   SRR36538378  SAMN54232216   
1  PX8

In [21]:
# Get genoflu results from Andersen and see if any fit

os.chdir(andersen)

genoflu_andersen = pd.read_csv("genoflu_results.tsv", delimiter="\t")
genoflu_andersen = genoflu_andersen.rename(columns={"sample":"SRA_Accession"}) # So we can merge with old
# genoflu_andersen["SRA_Accession"] = genoflu_andersen["Strain"] # So we can merge with new

# Find those genotyped by Andersen via merge
metadata_segments_known_andersen = metadata_segments_new.merge(genoflu_andersen, how="inner", on="SRA_Accession")
print(metadata_segments_known_andersen)
# Keep all known genotypes
metadata_segments_known = pd.merge(metadata_segments_old, metadata_segments_known_andersen, on="SRA_Accession", how="left") # Since we know both of these

print(metadata_segments_old)
print(metadata_segments_known_andersen)

       Accession GenBank_RefSeq         Assembly SRA_Accession     BioSample  \
0     PX880763.1        GenBank  GCA_054514015.1   SRR36538378  SAMN54232216   
1     PX880764.1        GenBank  GCA_054514015.1   SRR36538378  SAMN54232216   
2     PX880765.1        GenBank  GCA_054514015.1   SRR36538378  SAMN54232216   
3     PX880766.1        GenBank  GCA_054514015.1   SRR36538378  SAMN54232216   
4     PX880767.1        GenBank  GCA_054514015.1   SRR36538378  SAMN54232216   
...          ...            ...              ...           ...           ...   
3859  PQ012131.1        GenBank  GCA_040780295.1   SRR29281472  SAMN41656639   
3860  PQ012132.1        GenBank  GCA_040780295.1   SRR29281472  SAMN41656639   
3861  PQ012133.1        GenBank  GCA_040780295.1   SRR29281472  SAMN41656639   
3862  PQ012134.1        GenBank  GCA_040780295.1   SRR29281472  SAMN41656639   
3863  PQ012135.1        GenBank  GCA_040780295.1   SRR29281472  SAMN41656639   

        BioProject      Organism_Name  

### Create FASTA files of unknown genotypes 

In [19]:
metadata_segments_known

,Accession_x,GenBank_RefSeq_x,Assembly_x,SRA_Accession,BioSample_x,BioProject_x,Organism_Name_x,Species_x,Genus_x,Family_x,...,Date run_y,_merge,date,File Name,Genotype,"Genotype List Used, >=98.0%_y",Genotype Sample Title List_y,Genotype Percent Match List_y,Genotype Mismatch List_y,Genotype Average Depth of Coverage List_y
0,PX831491.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,PX831492.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PX831493.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,PX831494.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,PX831495.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89339,OK205667.1,GenBank,GCA_038971685.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89340,OK205668.1,GenBank,GCA_038971685.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89341,OK205669.1,GenBank,GCA_038971685.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89342,OK205670.1,GenBank,GCA_038971685.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# Create 8 fasta files per segment

# Get all the segments
metadata_segments_new["Partial_Header"] = metadata_segments_new["Assembly"].apply(lambda x: ">" + x if x == x else x) # .apply(lambda x: x.split("|")[-1].split(")")[0]) # Get only genbank name

print(metadata_segments_new["Partial_Header"].values[0:5])

# Create list of dataframes
df_list = []
for partial_header in list(set(metadata_segments_new["Partial_Header"].values)): # Unique partial headers only
    # Get smaller dataframe
    df = metadata_segments_new[metadata_segments_new["Partial_Header"] == partial_header]
    df = df.sort_values(by="Segment")
    # Make sure there are 8 segments
    if len(df) == 8:
        # Forbidden characters in headers
        for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]:
            df["Partial_Header"] = df["Partial_Header"].apply(lambda x: x.replace(c, "_"))
            
        df["full_header"] = df["Partial_Header"]
        df_list.append(df)

print(metadata_segments_new["Partial_Header"])

print(df_list[0]["full_header"].values[:10])

# Make fasta files
for df in df_list:
    segments = list(set(df["Segment"].apply(lambda x: int(x)).values))
    for segment in segments:
        one_row = df[df["Segment"] == segment]
        df_to_fasta(one_row, str(segment) + "_seg.fasta", temp_files)

['>GCA_054514015.1' '>GCA_054514015.1' '>GCA_054514015.1'
 '>GCA_054514015.1' '>GCA_054514015.1']
0         >GCA_054514015.1
1         >GCA_054514015.1
2         >GCA_054514015.1
3         >GCA_054514015.1
4         >GCA_054514015.1
                ...       
121771    >GCA_038165665.1
121772    >GCA_038165665.1
121773    >GCA_038165665.1
121774    >GCA_038165665.1
121775    >GCA_038165665.1
Name: Partial_Header, Length: 32432, dtype: object
['>GCA_051497195_1' '>GCA_051497195_1' '>GCA_051497195_1'
 '>GCA_051497195_1' '>GCA_051497195_1' '>GCA_051497195_1'
 '>GCA_051497195_1' '>GCA_051497195_1']


### Re-Labeling Using GenoFlu

**STOP HERE AND USE GENOFLU TO FIND NEW GENOTYPES.** Then, make sure "results.tsv" is in the downloads directory. <br>
To run GenoFLU-multi, first change directories to Multi-GenoFLU directory (and activate genoflu conda environment):
``` 
conda activate genoflu
cd GenoFLU-multi
```

And then call the python script:

``` 
python bin/genoflu-multi.py -f <FASTA_directory>
```

In [12]:
# Ensure that user does the above
input("Use genoflu. Afterwards, press ESCAPE to continue.")

''

In [13]:
# Merging

os.chdir(downloads_saved)

# Read in genoflu results
output_genoflu = pd.read_csv("results.tsv", delimiter="\t")

# Create partial headers to merge genoflu results with previously unknown segments
metadata_segments_new["Partial_Header_Merge"] = metadata_segments_new["Assembly"] # .apply(lambda x: ">" + x if x == x else x) # .apply(lambda x: x.replace(">", "")) # .apply(lambda x: x.split("|")[-1].split(")")[0]) # metadata_segments_new["Isolate"] #.apply(lambda x: x.split("|")[-1])
for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]: # Forbidden punctuation
    metadata_segments_new["Partial_Header_Merge"] = metadata_segments_new["Partial_Header_Merge"].apply(lambda x: x.replace(c, "_") if x==x else x)

# # Reformat so we can merge
# metadata_segments_new["Partial_Header_Merge"] = metadata_segments_new["Partial_Header_Merge"].apply(lambda x: x.replace(">", "")) # Removing header indicator
# metadata_segments_new["Partial_Header_Merge"] = metadata_segments_new["Partial_Header_Merge"].apply(lambda x: re.split(r'Influenza_A_virus__\D*_', x)[-1])
# metadata_segments_new["Partial_Header_Merge"] = metadata_segments_new["Partial_Header_Merge"].apply(lambda x: re.split(r'_(20|19).{2}_H.N.', x)[0])

# print(metadata_segments_new["Partial_Header_Merge"].values)

# Reformat so we can merge
output_genoflu["Partial_Header_Merge"] = output_genoflu["Strain"] #.apply(lambda x: re.split(r'Influenza_A_virus__\D*__\D*_', x)[-1])
# output_genoflu["Partial_Header_Merge"] = output_genoflu["Partial_Header_Merge"].apply(lambda x: re.split(r'_H.N._(20|19).{2}_.{2}_.{2}', x)[0])

# print(output_genoflu["Partial_Header_Merge"].values)
# metadata_segments_new["Strain"] = metadata_segments_new["Partial_Header"] # Renaming so we can merge

# Merge
metadata_genoflu = metadata_segments_new.merge(output_genoflu, how="inner", on="Partial_Header_Merge") #, suffixes=('_left', '_right')) 

# print(metadata_genoflu.dropna(subset="Genotype"))

# Fill the rest of the 8 segments with the same genotype
metadata_genoflu = metadata_genoflu.ffill(limit_area="inside")



print(metadata_genoflu)


        Accession GenBank_RefSeq         Assembly SRA_Accession     BioSample  \
0      PX880763.1        GenBank  GCA_054514015.1   SRR36538378  SAMN54232216   
1      PX880764.1        GenBank  GCA_054514015.1   SRR36538378  SAMN54232216   
2      PX880765.1        GenBank  GCA_054514015.1   SRR36538378  SAMN54232216   
3      PX880766.1        GenBank  GCA_054514015.1   SRR36538378  SAMN54232216   
4      PX880767.1        GenBank  GCA_054514015.1   SRR36538378  SAMN54232216   
...           ...            ...              ...           ...           ...   
31291  OK205883.1        GenBank  GCA_038165665.1           NaN           NaN   
31292  OK205884.1        GenBank  GCA_038165665.1           NaN           NaN   
31293  OK205885.1        GenBank  GCA_038165665.1           NaN           NaN   
31294  OK205886.1        GenBank  GCA_038165665.1           NaN           NaN   
31295  OK205887.1        GenBank  GCA_038165665.1           NaN           NaN   

         BioProject      Or

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\5\ipykernel_36384\1701310832.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  metadata_genoflu = metadata_genoflu.ffill(limit_area="inside")


### Concatenate with known genotypes

In [14]:
# Rename columns so we can concatenate
print(metadata_segments_known.columns)

metadata_segments_known = metadata_segments_known.reset_index(drop=True)
print(metadata_segments_known)

print(metadata_genoflu.columns)
metadata_segments_known["Genotype_official"] = metadata_segments_known["Genotype_y_y"]
metadata_segments_known["Serotype"] = metadata_segments_known["Genotype_x_x"]
metadata_genoflu["Genotype_official"] = metadata_genoflu["Genotype"]
metadata_genoflu["Serotype"] = metadata_genoflu["Genotype_x"]

metadata_genoflu = metadata_genoflu.reset_index(drop=True)
print(metadata_genoflu)

# Concatenation
metadata_genoflu_concat = pd.concat([metadata_segments_known, metadata_genoflu])

metadata_genoflu_concat

Index(['Accession_x', 'GenBank_RefSeq_x', 'Assembly_x', 'SRA_Accession',
       'BioSample_x', 'BioProject_x', 'Organism_Name_x', 'Species_x',
       'Genus_x', 'Family_x', 'Genotype_x_x', 'Isolate_x', 'Segment_x',
       'GenBank_Title_x', 'Length_x', 'Nuc_Completeness_x', 'Geo_Location_x',
       'Country_x', 'USA_x', 'Host_x', 'Tissue_Specimen_Source_x',
       'Submitters_x', 'Organization_x', 'Org_location_x', 'Publications_x',
       'Collection_Date_x', 'Release_Date_x', 'Molecule_type_x', 'size_x',
       'full_header_x', 'sequence_x', 'Partial_Header_temp_x', 'Strain_x',
       'Genotype_y_x', 'Genotype List Used, >=98.0%',
       'Genotype Sample Title List', 'Genotype Percent Match List',
       'Genotype Mismatch List', 'Genotype Average Depth of Coverage List',
       'Date run_x', 'Accession_y', 'GenBank_RefSeq_y', 'Assembly_y',
       'BioSample_y', 'BioProject_y', 'Organism_Name_y', 'Species_y',
       'Genus_y', 'Family_y', 'Genotype_x_y', 'Isolate_y', 'Segment_y',
   

,Accession_x,GenBank_RefSeq_x,Assembly_x,SRA_Accession,BioSample_x,BioProject_x,Organism_Name_x,Species_x,Genus_x,Family_x,...,Collection_Date,Release_Date,Molecule_type,size,full_header,sequence,Partial_Header_temp,Genotype_y,Partial_Header,Partial_Header_Merge
0,PX831491.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,PX831492.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PX831493.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,PX831494.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,PX831495.1,GenBank,GCA_054448965.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31291,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1995,2021-11-01,ssRNA(-),8.0,>OK205883.1 |Influenza A virus (A/chicken/Vera...,AGCAAAAGCAGGGGTATCAGATATCAAAATGGAAAGAATAGTGATT...,28159_398,NaN,>GCA_038165665.1,GCA_038165665_1
31292,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1995,2021-11-01,ssRNA(-),8.0,>OK205884.1 |Influenza A virus (A/chicken/Vera...,AGCAAAAGCAGGTTAGATAATCACTCACCGAGTGACATTCACATCA...,28159_398,NaN,>GCA_038165665.1,GCA_038165665_1
31293,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1995,2021-11-01,ssRNA(-),8.0,>OK205885.1 |Influenza A virus (A/chicken/Vera...,AGCAAAAGCAGGAGTGAAGATGAATCCAAATCAGAAGATAATAACA...,28159_398,NaN,>GCA_038165665.1,GCA_038165665_1
31294,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1995,2021-11-01,ssRNA(-),8.0,>OK205886.1 |Influenza A virus (A/chicken/Vera...,AGCAAAAGCAGGTAGATATTGAAAGATGAGTCTTCTAACCGAGGTC...,28159_398,NaN,>GCA_038165665.1,GCA_038165665_1


In [15]:
metadata_genoflu.dropna(subset="Accession")

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Strain_y,Genotype,"Genotype List Used, >=98.0%_y",Genotype Sample Title List_y,Genotype Percent Match List_y,Genotype Mismatch List_y,Genotype Average Depth of Coverage List_y,Date run_y,Genotype_official,Serotype
0,PX880763.1,GenBank,GCA_054514015.1,SRR36538378,SAMN54232216,PRJNA1207547,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,GCA_054514015_1,Not assigned: Only 7 segments >98.0% match fou...,"PB2:am27, PB1:ea3, HA:ea3, NP:am13, NA:am4N1, ...","am27:24-037288-001:PB2, ea3:22-013001-001:PB1,...","99.12%, 99.30%, 97.58%, 99.41%, 99.47%, 99.15%...","20, 16, 52, 10, 8, 12, 3, 8",Ran on FASTA - No Coverage Report,2026-01-22_11-57-25,Not assigned: Only 7 segments >98.0% match fou...,H5N1
1,PX880764.1,GenBank,GCA_054514015.1,SRR36538378,SAMN54232216,PRJNA1207547,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,GCA_054514015_1,Not assigned: Only 7 segments >98.0% match fou...,"PB2:am27, PB1:ea3, HA:ea3, NP:am13, NA:am4N1, ...","am27:24-037288-001:PB2, ea3:22-013001-001:PB1,...","99.12%, 99.30%, 97.58%, 99.41%, 99.47%, 99.15%...","20, 16, 52, 10, 8, 12, 3, 8",Ran on FASTA - No Coverage Report,2026-01-22_11-57-25,Not assigned: Only 7 segments >98.0% match fou...,H5N1
2,PX880765.1,GenBank,GCA_054514015.1,SRR36538378,SAMN54232216,PRJNA1207547,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,GCA_054514015_1,Not assigned: Only 7 segments >98.0% match fou...,"PB2:am27, PB1:ea3, HA:ea3, NP:am13, NA:am4N1, ...","am27:24-037288-001:PB2, ea3:22-013001-001:PB1,...","99.12%, 99.30%, 97.58%, 99.41%, 99.47%, 99.15%...","20, 16, 52, 10, 8, 12, 3, 8",Ran on FASTA - No Coverage Report,2026-01-22_11-57-25,Not assigned: Only 7 segments >98.0% match fou...,H5N1
3,PX880766.1,GenBank,GCA_054514015.1,SRR36538378,SAMN54232216,PRJNA1207547,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,GCA_054514015_1,Not assigned: Only 7 segments >98.0% match fou...,"PB2:am27, PB1:ea3, HA:ea3, NP:am13, NA:am4N1, ...","am27:24-037288-001:PB2, ea3:22-013001-001:PB1,...","99.12%, 99.30%, 97.58%, 99.41%, 99.47%, 99.15%...","20, 16, 52, 10, 8, 12, 3, 8",Ran on FASTA - No Coverage Report,2026-01-22_11-57-25,Not assigned: Only 7 segments >98.0% match fou...,H5N1
4,PX880767.1,GenBank,GCA_054514015.1,SRR36538378,SAMN54232216,PRJNA1207547,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,GCA_054514015_1,Not assigned: Only 7 segments >98.0% match fou...,"PB2:am27, PB1:ea3, HA:ea3, NP:am13, NA:am4N1, ...","am27:24-037288-001:PB2, ea3:22-013001-001:PB1,...","99.12%, 99.30%, 97.58%, 99.41%, 99.47%, 99.15%...","20, 16, 52, 10, 8, 12, 3, 8",Ran on FASTA - No Coverage Report,2026-01-22_11-57-25,Not assigned: Only 7 segments >98.0% match fou...,H5N1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31291,OK205883.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,GCA_038165665_1,Not assigned: Only 0 segments >98.0% match fou...,NaN,"am22:24-001196-005:PB2, am2:22-008760-007:PB1,...","91.49%, 95.47%, 90.89%, 79.86%, 93.92%, 88.50%...","194, 103, 194, 319, 91, 134, 39, 40",Ran on FASTA - No Coverage Report,2026-01-22_12-21-20,Not assigned: Only 0 segments >98.0% match fou...,H5N2
31292,OK205884.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,GCA_038165665_1,Not assigned: Only 0 segments >98.0% match fou...,NaN,"am22:24-001196-005:PB2, am2:22-008760-007:PB1,...","91.49%, 95.47%, 90.89%, 79.86%, 93.92%, 88.50%...","194, 103, 194, 319, 91, 134, 39, 40",Ran on FASTA - No Coverage Report,2026-01-22_12-21-20,Not assigned: Only 0 segments >98.0% match fou...,H5N2
31293,OK205885.1,GenBank,GCA_03

In [16]:
# Cut down to only columns we want
metadata_genoflu = metadata_genoflu[["Assembly", "GenBank_Title", "Host", "Collection_Date", "SRA_Accession", "Isolate", "Genotype_official", "Geo_Location", "full_header", "sequence", "Serotype", "Segment", "File Name", "Partial_Header"]] #, "Strain"]]

# Get genbank strain name
metadata_genoflu["genbank_name"] = metadata_genoflu["GenBank_Title"].apply(lambda x: x.split("(")[1] if x == x else x)
metadata_genoflu["Host"] = metadata_genoflu["genbank_name"].apply(lambda x: x.split("/")[1] if x == x else x)

metadata_genoflu = metadata_genoflu.dropna(subset="genbank_name") # [metadata_genoflu["Genotype"]  == "B3.13"]

KeyError: "['File Name'] not in index"

In [ ]:
metadata_genoflu

# for serotype in serotypes:
#     metadata_genoflu = metadata_genoflu[metadata_genoflu["Serotype"] == serotype]

,Assembly,GenBank_Title,Host,Collection_Date,SRA_Accession,Isolate,Genotype_official,Geo_Location,full_header,sequence,Serotype,Segment,File Name,Partial_Header,genbank_name
0,GCA_054514015.1,Influenza A virus (A/Peregrine Falcon/CA/25G04...,Peregrine Falcon,2025-11-20,SRR36538378,25G04838-001-original,Not assigned: Only 7 segments >98.0% match fou...,USA: CA,>PX880763.1 |Influenza A virus (A/Peregrine Fa...,ATGGATAGAATAAAAGAACTGAGAGATCTAATGTCACAGTCTCGCA...,H5N1,1.0,NaN,>GCA_054514015.1,A/Peregrine Falcon/CA/25G04838-001-original/2025
1,GCA_054514015.1,Influenza A virus (A/Peregrine Falcon/CA/25G04...,Peregrine Falcon,2025-11-20,SRR36538378,25G04838-001-original,Not assigned: Only 7 segments >98.0% match fou...,USA: CA,>PX880764.1 |Influenza A virus (A/Peregrine Fa...,ATGGATGTCAATCCGACTTTACTTTTCTTAAAAGTGCCAGCGCAAG...,H5N1,2.0,NaN,>GCA_054514015.1,A/Peregrine Falcon/CA/25G04838-001-original/2025
2,GCA_054514015.1,Influenza A virus (A/Peregrine Falcon/CA/25G04...,Peregrine Falcon,2025-11-20,SRR36538378,25G04838-001-original,Not assigned: Only 7 segments >98.0% match fou...,USA: CA,>PX880765.1 |Influenza A virus (A/Peregrine Fa...,ATGGAAGATTTTGTGCGACAATGCTTCAATCCAATGATCGTCGAGC...,H5N1,3.0,NaN,>GCA_054514015.1,A/Peregrine Falcon/CA/25G04838-001-original/2025
3,GCA_054514015.1,Influenza A virus (A/Peregrine Falcon/CA/25G04...,Peregrine Falcon,2025-11-20,SRR36538378,25G04838-001-original,Not assigned: Only 7 segments >98.0% match fou...,USA: CA,>PX880766.1 |Influenza A virus (A/Peregrine Fa...,ATGGAAAACATAGTACTTCTTCTTGCAATAATTAGCCTTGTTAAAA...,H5N1,4.0,NaN,>GCA_054514015.1,A/Peregrine Falcon/CA/25G04838-001-original/2025
4,GCA_054514015.1,Influenza A virus (A/Peregrine Falcon/CA/25G04...,Peregrine Falcon,2025-11-20,SRR36538378,25G04838-001-original,Not assigned: Only 7 segments >98.0% match fou...,USA: CA,>PX880767.1 |Influenza A virus (A/Peregrine Fa...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...,H5N1,5.0,NaN,>GCA_054514015.1,A/Peregrine Falcon/CA/25G04838-001-original/2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30067,GCA_039321425.1,Influenza A virus (A/Falco_rusticolus/EdoMex/C...,Falco_rusticolus,2022-10,NaN,CPA-19638-22,B3.2,Mexico: EdoMex,>OP691324.1 |Influenza A virus (A/Falco_rustic...,GGTTCACTCTGTCAAAATGGAGAACATAGTACTACTTCTTGCAATA...,H5N1,4.0,NaN,>GCA_039321425.1,A/Falco_rusticolus/EdoMex/CPA-19638-22/2022
30068,GCA_039321425.1,Influenza A virus (A/Falco_rusticolus/EdoMex/C...,Falco_rusticolus,2022-10,NaN,CPA-19638-22,B3.2,Mexico: EdoMex,>OP691325.1 |Influenza A virus (A/Falco_rustic...,AAGCAGGGTAGATAATCACTCACTGAGTGACATCCACATCATGGCG...,H5N1,5.0,NaN,>GCA_039321425.1,A/Falco_rusticolus/EdoMex/CPA-19638-22/2022
30069,GCA_039321425.1,Influenza A virus (A/Falco_rusticolus/EdoMex/C...,Falco_rusticolus,2022-10,NaN,CPA-19638-22,B3.2,Mexico: EdoMex,>OP691326.1 |Influenza A virus (A/Falco_rustic...,AAAGCAGGAGTTCAAAATGAATCCAAATCAAAAGATAACAACCATT...,H5N1,6.0,NaN,>GCA_039321425.1,A/Falco_rusticolus/EdoMex/CPA-19638-22/2022
30070,GCA_039321425.1,Influenza A virus (A/Falco_rusticolus/EdoMex/C...,Falco_rusticolus,2022-10,NaN,CPA-19638-22,B3.2,Mexico: EdoMex,>OP691327.1 |Influenza A virus (A/Falco_rustic...,AAAAGCAGGTAGATATTGAAAGATGAGTCTTCTAACCGAGGTCGAA...,H5N1,7.0,NaN,>GCA_039321425.1,A/Falco_rusticolus/EdoMex/CPA-19638-22/2022


## Relabeling Sequences

We want:
* Host
* Geo-Location
* Isolate
* Year
* Collection Date
* Host Type
* Genotype

After running the below code, **STOP TO CHECK** if any new animals appear

In [ ]:
# Animals 

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genoflu)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['aves', 'neotropic cormorant', 'ring-billed gull', 'owl', 'fish crow', 'striped skunk', 'pluvialis dominica', 'mergus', 'environment', 'swine', 'dunlin', 'black swan', 'mottled duck', 'royal tern', 'gallus gallus domesticus', 'gannet', 'snowy owl', 'great egret', 'common loon', 'western grebe', 'turkey vulture', 'gallus gallus', 'nasua nasua', 'skunk', 'pelican', 'harbor seal', 'american green-winged teal', 'parasitic jaeger', 'ring-necked duck', 'eagle', 'eurasian collared dove', 'northern shoveler', 'serval', 'avian', 'horned grebe', 'barred owl', 'american wigeon', 'magpie', 'grackle', 'megascops choliba', 'eastern screech owl', 'willet', 'parrot', 'sandwich tern', 'osprey', 'western screech owl', 'antofagasta', 'baikal teal', 'pinniped', 'american wood stork', 'south american sea lion', 'canada goose', 'pelecanus thagus', 'lesser snow goose white-morph', 'pintail', 'vulpes vulpes', 'peafowl', 'sandhill crane', 'ring-necked pheasant', 'glaucous gull', 'green winged teal', 'black tu

In [ ]:
input("Check animals output. Afterwards, press ESCAPE to continue.")

In [ ]:
# Re-label sequences with no assigned genotype as "Unassigned"

metadata_genoflu["Genotype_official"] = metadata_genoflu["Genotype_official"].apply(lambda x: "Unassigned" if "Not assigned" in str(x) else x) # Unassigned segments are labeled "Unassigned"
metadata_genoflu["Genotype"] = metadata_genoflu["Genotype_official"] # Rename column again to not break old code

In [ ]:
metadata_genoflu

,Accession,GenBank_Title,Host,Collection_Date,SRA_Accession,Isolate,Genotype_official,Geo_Location,full_header,sequence,Serotype,Segment,File Name,Partial_Header,Strain,genbank_name,Genotype
0,PV602142.1,Influenza A virus (A/American green-winged tea...,American green-winged teal,2024-10-17,NaN,IZ24_0776,D1.1,USA: Alaska,>Influenza A virus |USA: Alaska|IZ24_0776|H5N1...,GGTTCACTCTGTCAAAATGGAAAACATAGTACTTCTTCTTGCAATA...,H5N1,4,NaN,Influenza_A_virus__USA__Alaska_IZ24_0776_H5N1_...,Influenza_A_virus__USA__Alaska_IZ24_0776_H5N1_...,A/American green-winged teal/USA/IZ24_0776/2024,D1.1
1,PV602143.1,Influenza A virus (A/American green-winged tea...,American green-winged teal,2024-10-17,NaN,IZ24_0776,D1.1,USA: Alaska,>Influenza A virus |USA: Alaska|IZ24_0776|H5N1...,TAGATATTGAAAGATGAGTCTTCTAACCGAGGTCGAAACGTACGTT...,H5N1,7,NaN,Influenza_A_virus__USA__Alaska_IZ24_0776_H5N1_...,Influenza_A_virus__USA__Alaska_IZ24_0776_H5N1_...,A/American green-winged teal/USA/IZ24_0776/2024,D1.1
2,PV602144.1,Influenza A virus (A/American green-winged tea...,American green-winged teal,2024-10-17,NaN,IZ24_0776,D1.1,USA: Alaska,>Influenza A virus |USA: Alaska|IZ24_0776|H5N1...,AAAGCAGGAGTTCAAAATGAATCCAAATCAAAAGATAATAACTATC...,H5N1,6,NaN,Influenza_A_virus__USA__Alaska_IZ24_0776_H5N1_...,Influenza_A_virus__USA__Alaska_IZ24_0776_H5N1_...,A/American green-winged teal/USA/IZ24_0776/2024,D1.1
3,PV602145.1,Influenza A virus (A/American green-winged tea...,American green-winged teal,2024-10-17,NaN,IZ24_0776,D1.1,USA: Alaska,>Influenza A virus |USA: Alaska|IZ24_0776|H5N1...,GTAGATAATCACTCACTGAGTGACATCCACATCATGGCGTCTCAAG...,H5N1,5,NaN,Influenza_A_virus__USA__Alaska_IZ24_0776_H5N1_...,Influenza_A_virus__USA__Alaska_IZ24_0776_H5N1_...,A/American green-winged teal/USA/IZ24_0776/2024,D1.1
4,PV602146.1,Influenza A virus (A/American green-winged tea...,American green-winged teal,2024-10-17,NaN,IZ24_0776,D1.1,USA: Alaska,>Influenza A virus |USA: Alaska|IZ24_0776|H5N1...,GTGACAAAAACATAATGGATTCCAACACTGTGTCAAGCTTTCAGGT...,H5N1,8,NaN,Influenza_A_virus__USA__Alaska_IZ24_0776_H5N1_...,Influenza_A_virus__USA__Alaska_IZ24_0776_H5N1_...,A/American green-winged teal/USA/IZ24_0776/2024,D1.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67619,OK205883.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,NaN,28159-398,Unassigned,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGGGTATCAGATATCAAAATGGAAAGAATAGTGATT...,H5N2,4,NaN,Influenza_A_virus__Mexico__Veracruz_28159_398_...,NaN,A/chicken/Veracruz/28159-398/1995,Unassigned
67620,OK205884.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,NaN,28159-398,Unassigned,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGTTAGATAATCACTCACCGAGTGACATTCACATCA...,H5N2,5,NaN,Influenza_A_virus__Mexico__Veracruz_28159_398_...,NaN,A/chicken/Veracruz/28159-398/1995,Unassigned
67621,OK205885.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,NaN,28159-398,Unassigned,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGAGTGAAGATGAATCCAAATCAGAAGATAATAACA...,H5N2,6,NaN,Influenza_A_virus__Mexico__Veracruz_28159_398_...,NaN,A/chicken/Veracruz/28159-398/1995,Unassigned
67622,OK205886.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,NaN,28159-398,Unassigned,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGTAGATATTGAAAGATGAGTCTTCTAACCGAGGTC...,H5N2,7,NaN,Influenza_A_virus__Mexico__Veracruz_28159_398_...,NaN,A/chicken/Veracruz/28159-398/1995,Unassigned


In [ ]:
# Re-Labeling

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Make sure NaN doesn't mess up the whole name
metadata_genoflu = metadata_genoflu.fillna("")
metadata_genoflu["Host"] = metadata_genoflu["Host"].apply(str.lower)

# Fix animals
fix_animals_andersen(metadata_genoflu, animals_ref)

# Get the years
metadata_genoflu["Years"] = metadata_genoflu["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y"))

# Get geographic locations
metadata_genoflu["Geo_Location_Abrv"] = metadata_genoflu["Geo_Location"].apply(lambda x: 
                                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                                        states_ref.loc[states_ref["Abbreviation"].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
                                                                        + "-" + 
                                                                        x.split(" ")[-1]
                                                                        if states_ref["Abbreviation"].str.contains("|".join((re.sub(":? ", ",", x).replace(" ", "_").split(','))), regex=True).any()
                                                                        # If "x" has the full state name (e.g. "Maryland")
                                                                        else states_ref.loc[states_ref['State'].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
                                                                        + "-" + 
                                                                        states_ref.loc[states_ref['State'].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
                                                                        if states_ref["State"].str.contains("|".join((re.sub(":? ", ",", x).replace(" ", "_").split(','))), regex=True).any()
                                                                        # If "x" has neither the state abbreviation nor the full state name nor is "USA"
                                                                        else 
                                                                        x
                                                                        )

metadata_genoflu["Geo_Location_Abrv"] = metadata_genoflu["Geo_Location_Abrv"].apply(lambda x: x.split("-")[0] if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] else x)


print(metadata_genoflu["Geo_Location_Abrv"])

0                 USA-AK
1                 USA-AK
2                 USA-AK
3                 USA-AK
4                 USA-AK
              ...       
67619    Mexico-Veracruz
67620    Mexico-Veracruz
67621    Mexico-Veracruz
67622    Mexico-Veracruz
67623    Mexico-Veracruz
Name: Geo_Location_Abrv, Length: 67800, dtype: object


In [ ]:
# If there is no SRA Accession, replace identifier with Accession
metadata_genoflu['SRA_Accession'] = np.where(metadata_genoflu['SRA_Accession'] == "", metadata_genoflu['Assembly'].apply(lambda x: x.split(".")[0]), metadata_genoflu['SRA_Accession'])
# If there is no Assembly, replace identifier with Accession -- only for PB2
metadata_genoflu["Identifier"] = np.where(metadata_genoflu["SRA_Accession"] == "", metadata_genoflu["Accession"].apply(lambda x: x.split(".")[0] if x["Segment"] == 1 else np.nan), metadata_genoflu["SRA_Accession"])
# Fill in other nans with PB2 Accession (interpolate, maximum of 7 other sequences)
metadata_genoflu["Identifier"] = metadata_genoflu.ffill(metadata_genoflu["Identifier"], limit=7, limit_area="inside")

# Make new labels
names = ">" + metadata_genoflu["Identifier"].apply(lambda x: x.split(",")[0]) + "|" + metadata_genoflu["genbank_name"] + "|" + metadata_genoflu["Serotype"] + "|" + metadata_genoflu["Geo_Location_Abrv"] + "|" + metadata_genoflu["Collection_Date"] + "|" + metadata_genoflu["Host_Type"] + "|" + metadata_genoflu["Genotype"]

metadata_genoflu["Name"] = names


print(metadata_genoflu)

        Accession                                      GenBank_Title  \
0      PV602142.1  Influenza A virus (A/American green-winged tea...   
1      PV602143.1  Influenza A virus (A/American green-winged tea...   
2      PV602144.1  Influenza A virus (A/American green-winged tea...   
3      PV602145.1  Influenza A virus (A/American green-winged tea...   
4      PV602146.1  Influenza A virus (A/American green-winged tea...   
...           ...                                                ...   
67619  OK205883.1  Influenza A virus (A/chicken/Veracruz/28159-39...   
67620  OK205884.1  Influenza A virus (A/chicken/Veracruz/28159-39...   
67621  OK205885.1  Influenza A virus (A/chicken/Veracruz/28159-39...   
67622  OK205886.1  Influenza A virus (A/chicken/Veracruz/28159-39...   
67623  OK205887.1  Influenza A virus (A/chicken/Veracruz/28159-39...   

                             Host Collection_Date SRA_Accession    Isolate  \
0      american green-winged teal      2024-10-17      PV

In [ ]:
os.chdir(complete_files)

# De-duplicate

print(len(metadata_genoflu))
metadata_genoflu = metadata_genoflu.drop_duplicates(subset=["Isolate", "genbank_name", "Segment"], keep="last")
print(len(metadata_genoflu))

# Save metadata
metadata_genoflu.to_csv("NCBI_Virus_" + date_range + "_metadata.csv") # Make file for metadata


67800
67696


## Rename segments and make complete FASTA files

In [ ]:
# Set up segments

if len(genotypes) > 3: # If we're not doing maintenance only
    genotypes.append("Unassigned") # Make sure unassigned genotypes are included
segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"} # Name segments 
metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].apply(lambda x: int(x)).map(segments)

# Separate into several dataframes based on genotype + segment
segment_genotype_dfs = []
for segment in segments.values():
    m_g = metadata_genoflu[metadata_genoflu["Segment_Name"] == segment]
    for genotype in list(set(m_g["Genotype"].values)):
        if genotype in genotypes:
            df = m_g[(m_g["Genotype"] == genotype)] 
            # df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)
            segment_genotype_dfs.append(df)
            pair = genotype + "_" + segment
            print(pair)

Minor104_PB2
Minor14_PB2
B1.1_PB2
Minor04_PB2
B3.7_PB2
A2_PB2
B5.1_PB2
Minor11_PB2
Minor08_PB2
Minor01_PB2
Minor07_PB2
Minor34_PB2
Minor12_PB2
B1.2_PB2
Minor94_PB2
B4.1_PB2
B1.3_PB2
A6_PB2
A4_PB2
B2.2_PB2
Minor50_PB2
B2.1_PB2
C2.1_PB2
B3.5_PB2
A3_PB2
D1.1_PB2
A1_PB2
Unassigned_PB2
B3.6_PB2
B3.13_PB2
A5_PB2
B3.1_PB2
D1.3_PB2
D1.2_PB2
Minor105_PB2
Minor19_PB2
Minor13_PB2
B3.12_PB2
Minor28_PB2
Minor102_PB2
B3.2_PB2
Minor98_PB2
Minor45_PB2
B3.3_PB2
B3.4_PB2
Minor09_PB2
Minor104_PB1
Minor14_PB1
B1.1_PB1
Minor04_PB1
B3.7_PB1
A2_PB1
B5.1_PB1
Minor11_PB1
Minor08_PB1
Minor01_PB1
Minor07_PB1
Minor34_PB1
Minor12_PB1
B1.2_PB1
Minor94_PB1
B4.1_PB1
B1.3_PB1
A6_PB1
A4_PB1
B2.2_PB1
Minor50_PB1
B2.1_PB1
C2.1_PB1
B3.5_PB1
A3_PB1
D1.1_PB1
A1_PB1
Unassigned_PB1
B3.6_PB1
B3.13_PB1
A5_PB1
B3.1_PB1
D1.3_PB1
D1.2_PB1
Minor105_PB1
Minor19_PB1
Minor13_PB1
B3.12_PB1
Minor28_PB1
Minor102_PB1
B3.2_PB1
Minor98_PB1
Minor45_PB1
B3.3_PB1
B3.4_PB1
Minor09_PB1
Minor104_PA
Minor14_PA
B1.1_PA
Minor04_PA
B3.7_PA
A2_PA
B5.1

In [ ]:
# Create FASTA files

os.chdir(complete_files)

for df in segment_genotype_dfs:
    print(df)
    
    df = df.reset_index()
    if len(df["Genotype"].values[0]) > 0: # If we have assigned genotypes, including "Unassigned"
        file_name = df["Genotype"].values[0] + "_" + df["Segment_Name"].values[0] + "_" + date_range + ".fasta"
        output_file = open(complete_files + file_name, "w")

        for index, row in df.iterrows():
            name = df.loc[index, "Name"]
            name = name.replace(" ", "_")
            # names.append(name)
            sequence = df.loc[index, "sequence"]
            # First is header, second is sequence
            output_file.write(name + "\n")
            output_file.write(sequence + "\n")
        
    output_file.close()

        Accession                                      GenBank_Title  \
12008  PV267613.1  Influenza A virus (A/snow goose/Arkansas/AR24-...   

             Host Collection_Date SRA_Accession   Isolate Genotype_official  \
12008  snow goose      2024-12-06      PV267613  AR24-104          Minor104   

        Geo_Location                                        full_header  \
12008  USA: Arkansas  >Influenza A virus |USA: Arkansas|AR24-104|H5N...   

                                                sequence  ... File Name  \
12008  TCAAATATATTCAATATGGAGAGAATAAAAGAACTAAGAGATCTAA...  ...             

                                          Partial_Header Strain  \
12008  Influenza_A_virus__USA__Arkansas_AR24_104_H5N1...          

                              genbank_name  Genotype   Host_Type Years  \
12008  A/snow goose/Arkansas/AR24-104/2024  Minor104  wild_avian  2024   

      Geo_Location_Abrv                                               Name  \
12008            USA-AR  >PV2676